First some setup to make data for our prediction model

In [2]:
import numpy as np
import pandas as pd
import sklearn
# import librosa
import tensorflow as tf
from tensorflow.keras.layers import Dropout, Dense, Input, BatchNormalization, Activation
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ReduceLROnPlateau
# import soundfile as sf
# from IPython.display import Audio, display
import os

In [3]:
FRAME_LENGTH = 1024
HOP_LENGTH = 512
SAMPLE_RATE = 32000

In [49]:
def extract_features(signal):
    zero_crossing_rate = librosa.feature.zero_crossing_rate(y=signal, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
    mfcc20 = librosa.feature.mfcc(y=signal, n_mfcc=20, sr=SAMPLE_RATE)
    spectral_rolloff = librosa.feature.spectral_rolloff(y=signal, sr=SAMPLE_RATE, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
    return (zero_crossing_rate, mfcc20, spectral_rolloff)    
    
def slice_audio(duration, input):
    audio_array, sample_rate = librosa.load(input, sr=SAMPLE_RATE)
    seconds_per_sample = 1/sample_rate
    samples_for_duration = duration/seconds_per_sample
    sliced_audio = audio_array[0:int(samples_for_duration)]    
    return sliced_audio

def normalize(array):
    min_val = np.min(array)
    max_val = np.max(array)
    
    if max_val == min_val:
        return np.zeros_like(array)  # or return array, depending on your use case
    
    return (array - min_val) / (max_val - min_val)

In [44]:
metadata_df = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train.csv")


In [ ]:
giant_df_list = []
for primary_label, scientific_name, class_name in zip(metadata_df['primary_label'], metadata_df['scientific_name'], metadata_df['class_name']):
    dir_path = "/kaggle/input/competitions/birdclef-2026/train_audio/"+str(primary_label)    
    if (os.path.isdir(dir_path)):
        for filename in os.listdir(dir_path):
            slice = slice_audio(6, dir_path+"/"+str(filename))
            
            if (slice.size < 64000): continue # if less than 2 seconds, probably not worth considering
            frames = int(slice.size/HOP_LENGTH)
            classname = [class_name for i in range(frames)]
            scientificname = [scientific_name for i in range(frames)]
            primarylabel = [primary_label for i in range(frames)]
            data = {"primary_label": primarylabel, "scientific_name": scientificname, "class_name": classname}
            
            (zero_crossing_rate, mfcc20, spectral_rolloff) = extract_features(slice)
            
            feature_df = pd.DataFrame(data=data)
            feature_df["zero_crossing_rate"] = normalize(zero_crossing_rate)[:frames]
            feature_df["spectral_rolloff"] = normalize(spectral_rolloff)[:frames]
    
            # I use mfcc 1-20. In my previous research, mfcc 1-5 and mfcc 14 have the greatest importance
            
            for i in range(len(mfcc20)):
                feature = "mfcc"+str(i+1)
                feature_df[feature] = normalize(mfcc20[i])[:frames]
    
            giant_df_list.append(feature_df)

In [ ]:
df = pd.concat(giant_df_list)
df.to_csv("/kaggle/working/features.csv", index=False)

Now that we have our data we can make the model. I saved the above dataset to kaggle to save time in later development.

In [4]:
df = pd.read_csv("/kaggle/input/datasets/pierce3a44/birdcleffeatures/features.csv", dtype={"primary_label":str})
X = df.drop(["primary_label", "scientific_name", "class_name"], axis=1)
Y = df['primary_label'].astype('category').cat.codes.values
X_train, X_test, Y_train, Y_test = sklearn.model_selection.train_test_split(X, Y, test_size=0.2, random_state=10)

In [6]:
mlp_model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(1024),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.4),
    Dense(512),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.3),
    Dense(206, activation='softmax')
])

mlp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp_model.summary()

I0000 00:00:1775092442.633753      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1024)           │        23,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 206)            │       105,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 660,174 (2.52 MB)

 Trainable params: 657,102 (2.51 MB)

 Non-trainable params: 3,072 (12.00 KB)

In [ ]:
history = mlp_model.fit(X_train, Y_train, validation_data=(X_test, Y_test), 
                        epochs=100, batch_size=1024)

In [10]:
mlp_model.save_weights('/kaggle/working/checkpoint_mlp2.weights.h5')


In [19]:
predictions = mlp_model.predict(tf.constant(X_test.iloc[0].values.reshape(1, -1)))
# dictionary = {}
print(len(predictions[0]))
dictionary = {}
unique_labels = list(dict.fromkeys(df["primary_label"]))
for i in range(len(predictions[0])):
    dictionary[unique_labels[i]] = predictions[0][i]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
206


In [ ]:
highest1 = max(dictionary.items(), key=lambda x: x[1])
print(highest1)
dictionary.pop(highest1[0])
highest2 = max(dictionary.items(), key=lambda x: x[1])
print(highest2)
dictionary.pop(highest2[0])
highest3 = max(dictionary.items(), key=lambda x: x[1])
print(highest3)
highest4 = max(dictionary.items(), key=lambda x: x[1])
print(highest4)
dictionary.pop(highest4[0])
highest5 = max(dictionary.items(), key=lambda x: x[1])
print(highest5)

cat_series = df['primary_label'].astype('category')
labels = cat_series.cat.categories

print(labels[Y_test[0]])